# DEASUS · Documentos de Acompanhamento Periódico do Auditor

1. No BI, aba **Acompanhamento Periódico**: filtre o período → na tabela, menu **…** → **Exportar dados** → **Dados com layout atual** → `.xlsx`.
2. Aqui: menu **Ambiente de execução → Executar tudo** e use o **painel** que aparece abaixo.

O modelo do documento já está embutido. Para usar outro modelo, escolha o `.docx` junto com o Excel.

Dica: para levar a avaliação descritiva pronta, acrescente no Excel uma coluna **Avaliação Descritiva**.

In [ ]:
#@title ▶ Painel (clique em Executar tudo) { display-mode: "form" }
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "docxtpl"], check=False)
# código: gerar_documentos.py e painel.py (repositório BI-Interno-DEASUS, pasta gerador_documentos)
open("gerar_documentos.py", "w", encoding="utf-8").write('"""Gera os Documentos de Acompanhamento Periódico do Auditor (um .docx por auditor).\n\nEntrada:\n  1. O Excel exportado da tabela da aba "Acompanhamento Periódico" do BI\n     (menu ... da tabela -> Exportar dados -> Dados com layout atual -> .xlsx).\n  2. O modelo .docx com os campos {{ Auditor }}, {{ Quadrimestre }}, {{ Avaliadores }},\n     {{ atividade_1 }} ... {{ avaliacao_5 }}, {{ proas }} e {{ Avaliacao_Descritiva }}.\n\nAs colunas são lidas PELO NOME (não pela posição). Se alguém renomear uma coluna no BI,\najuste a lista ATIVIDADES abaixo.\n\nUso no computador (opcional; no dia a dia use o notebook no Colab):\n    python gerar_documentos.py exportacao.xlsx modelo.docx --periodo "1º/2026" --avaliadores "Fulano, Beltrano"\n"""\nimport os\nimport re\nimport ast\nimport zipfile\nimport unicodedata\nfrom datetime import datetime\n\nimport pandas as pd\nfrom docxtpl import DocxTemplate\n\n# Linhas da tabela do documento: (texto que aparece no documento, coluna do Excel exportado).\n# O modelo tem 5 linhas (atividade_1..5). Para mudar o que aparece, troque aqui.\nATIVIDADES = [\n    ("Total de auditorias designadas", "Total de auditorias designadas"),\n    ("Auditorias (coordenação)", "Auditorias (coordenação)"),\n    ("Auditorias (composição de equipe)", "Auditorias (composição de equipe)"),\n    ("Relatórios Preliminares entregues", "Relatórios Preliminares entregues"),\n    ("Qtd de Matrizes RAI", "Qtd de Matrizes RAI"),   # 5ª linha igual à do notebook antigo\n]\nCOL_AUDITOR = "Auditor"\nCOL_PROAS = "PROAs"\nCOL_DESCRITIVA = "Avaliação Descritiva"   # opcional: se existir no Excel, vai para o documento\n\nMESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",\n         "agosto", "setembro", "outubro", "novembro", "dezembro"]\n\n\ndef _chave(txt):\n    """Compara nomes de coluna sem acento, maiúsculas ou espaços extras."""\n    txt = unicodedata.normalize("NFD", str(txt))\n    txt = "".join(c for c in txt if unicodedata.category(c) != "Mn")\n    return re.sub(r"\\s+", " ", txt).strip().lower()\n\n\ndef ler_exportacao(caminho):\n    """Lê o Excel/CSV exportado do BI. Acha a linha de cabeçalho (a que tem \'Auditor\')\n    e descarta linhas vazias, de total e as notas de filtro que o Power BI põe no fim."""\n    if caminho.lower().endswith(".csv"):\n        bruto = pd.read_csv(caminho, header=None, dtype=object)\n    else:\n        bruto = pd.read_excel(caminho, header=None, dtype=object)\n    linha_cab = None\n    for i, linha in bruto.iterrows():\n        if any(_chave(v) == _chave(COL_AUDITOR) for v in linha.values if pd.notna(v)):\n            linha_cab = i\n            break\n    if linha_cab is None:\n        raise ValueError(f"Não achei a coluna \'{COL_AUDITOR}\' no arquivo {caminho}. "\n                         "Confira se exportou a tabela da aba Acompanhamento Periódico.")\n    df = bruto.iloc[linha_cab + 1:].copy()\n    df.columns = [str(c).strip() if pd.notna(c) else f"_col{j}" for j, c in enumerate(bruto.iloc[linha_cab])]\n    mapa = {_chave(c): c for c in df.columns}\n    df = df.rename(columns={mapa[_chave(COL_AUDITOR)]: COL_AUDITOR})\n    df[COL_AUDITOR] = df[COL_AUDITOR].astype("string").str.strip()\n    ruido = df[COL_AUDITOR].isna() | (df[COL_AUDITOR] == "") | df[COL_AUDITOR].str.lower().isin(["total"]) \\\n        | df[COL_AUDITOR].str.lower().str.startswith(("filtros aplicados", "applied filters"))\n    return df[~ruido].reset_index(drop=True), mapa\n\n\ndef _numero(v):\n    if v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and not v.strip()):\n        return "0"\n    try:\n        f = float(str(v).replace(",", "."))\n        return str(int(f)) if f.is_integer() else f"{f:.2f}".replace(".", ",")\n    except ValueError:\n        return str(v)\n\n\ndef _proas(v):\n    """Aceita \'A; B; C\' (BI) ou "[\'A\', \'B\']" (planilha antiga do notebook)."""\n    if v is None or (isinstance(v, float) and pd.isna(v)):\n        return ""\n    s = str(v).strip()\n    if s.startswith("["):\n        try:\n            itens = ast.literal_eval(s)\n        except (ValueError, SyntaxError):\n            itens = [s]\n    else:\n        itens = re.split(r";|\\n", s)\n    return "\\n".join(re.sub(r"\\s+", " ", i).strip() for i in itens if str(i).strip())\n\n\ndef montar_contexto(linha, mapa, periodo, avaliadores):\n    hoje = datetime.now()\n    ctx = {\n        "Auditor": linha[COL_AUDITOR],\n        "Quadrimestre": periodo,\n        "Avaliadores": avaliadores,\n        "DataHoje": f"{hoje.day} de {MESES[hoje.month - 1]} de {hoje.year}",\n        "proas": _proas(linha.get(mapa.get(_chave(COL_PROAS)))),\n        "Avaliacao_Descritiva": "",\n    }\n    col_desc = mapa.get(_chave(COL_DESCRITIVA))\n    if col_desc and pd.notna(linha.get(col_desc)):\n        ctx["Avaliacao_Descritiva"] = str(linha.get(col_desc))\n    for i, (rotulo, coluna) in enumerate(ATIVIDADES, start=1):\n        col = mapa.get(_chave(coluna))\n        ctx[f"atividade_{i}"] = rotulo\n        ctx[f"avaliacao_{i}"] = _numero(linha.get(col)) if col else ""\n    return ctx\n\n\ndef gerar(caminho_excel, caminho_modelo, periodo, avaliadores="", pasta_saida="documentos_acompanhamento"):\n    df, mapa = ler_exportacao(caminho_excel)\n    faltando = [c for _, c in ATIVIDADES if _chave(c) not in mapa]\n    if faltando:\n        print("ATENÇÃO: colunas não encontradas no Excel (ficarão em branco no documento):", faltando)\n    if _chave(COL_PROAS) not in mapa:\n        print(f"ATENÇÃO: coluna \'{COL_PROAS}\' não encontrada; a lista de PROAs sairá vazia.")\n    os.makedirs(pasta_saida, exist_ok=True)\n    arquivos = []\n    for n, linha in df.iterrows():\n        ctx = montar_contexto(linha, mapa, periodo, avaliadores)\n        doc = DocxTemplate(caminho_modelo)\n        doc.render(ctx)\n        nome = re.sub(r"[^\\w\\-]+", "_", ctx["Auditor"]).strip("_")\n        caminho = os.path.join(pasta_saida, f"acompanhamento_{n + 1:02d}_{nome}.docx")\n        doc.save(caminho)\n        arquivos.append(caminho)\n        print("Gerado:", caminho)\n    zip_path = pasta_saida.rstrip("/\\\\") + ".zip"\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:\n        for a in arquivos:\n            z.write(a, os.path.basename(a))\n    print(f"\\n{len(arquivos)} documento(s) gerado(s). ZIP: {zip_path}")\n    return zip_path\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("excel")\n    ap.add_argument("modelo")\n    ap.add_argument("--periodo", required=True, help=\'Ex.: "1º/2026"\')\n    ap.add_argument("--avaliadores", default="")\n    ap.add_argument("--saida", default="documentos_acompanhamento")\n    a = ap.parse_args()\n    gerar(a.excel, a.modelo, a.periodo, a.avaliadores, a.saida)\n')
open("painel.py", "w", encoding="utf-8").write('"""Painel com botões (ipywidgets) para o gerador de documentos. Usado pelo notebook do Colab.\n\nO modelo .docx vem embutido no notebook (MODELO_PADRAO_B64). Se a pessoa enviar um .docx\njunto com o Excel, ele substitui o modelo embutido.\n"""\nimport base64\nimport io\nimport os\nimport re\nimport shutil\nimport contextlib\n\nimport ipywidgets as w\nfrom IPython.display import display\n\nimport gerar_documentos as g\n\nPASTA_ENTRADA = "entrada"\n\n\ndef _arquivos_enviados(valor):\n    """Normaliza o valor do FileUpload (ipywidgets 7: dict; ipywidgets 8: tupla) -> [(nome, bytes)]."""\n    if isinstance(valor, dict):\n        return [(nome, bytes(info["content"])) for nome, info in valor.items()]\n    return [(item["name"], bytes(item["content"])) for item in valor]\n\n\ndef criar_painel(modelo_padrao_b64=None, baixar=None):\n    """Monta e exibe o painel. `baixar(caminho)` é chamado para baixar o ZIP (no Colab: files.download)."""\n    estilo = {"description_width": "110px"}\n    larg = w.Layout(width="420px")\n    periodo = w.Text(description="Período:", placeholder="ex.: 1º/2026", style=estilo, layout=larg)\n    avaliadores = w.Text(description="Avaliadores:", placeholder="ex.: Fulano e Beltrano", style=estilo, layout=larg)\n    upload = w.FileUpload(accept=".xlsx,.csv,.docx", multiple=True, description="1. Escolher Excel do BI",\n                          layout=w.Layout(width="260px"))\n    recebidos = w.HTML("<i>Nenhum arquivo escolhido.</i>")\n    gerar_btn = w.Button(description="2. Gerar documentos", button_style="primary", icon="cogs",\n                         layout=w.Layout(width="260px"), disabled=True)\n    baixar_btn = w.Button(description="3. Baixar ZIP", button_style="success", icon="download",\n                          layout=w.Layout(width="260px"), disabled=True)\n    link = w.HTML()\n    log = w.Output(layout=w.Layout(border="1px solid #D5DCE6", padding="6px", max_height="260px", overflow="auto"))\n    estado = {"zip": None}\n\n    def ao_escolher(_=None):\n        nomes = [n for n, _ in _arquivos_enviados(upload.value)]\n        excel = [n for n in nomes if n.lower().endswith((".xlsx", ".csv"))]\n        modelo = [n for n in nomes if n.lower().endswith(".docx")]\n        partes = []\n        partes.append(f"Excel: <b>{excel[-1]}</b>" if excel else "<span style=\'color:#D64545\'>Falta o Excel exportado do BI</span>")\n        partes.append(f"Modelo: <b>{modelo[-1]}</b>" if modelo else "Modelo: <b>padrão (embutido)</b>")\n        recebidos.value = " · ".join(partes)\n        gerar_btn.disabled = not excel\n        baixar_btn.disabled = True\n        link.value = ""\n\n    def ao_gerar(_=None):\n        log.clear_output()\n        baixar_btn.disabled = True\n        link.value = ""\n        with log:\n            if not periodo.value.strip():\n                print("Preencha o Período antes de gerar.")\n                return\n            shutil.rmtree(PASTA_ENTRADA, ignore_errors=True)\n            shutil.rmtree("documentos_acompanhamento", ignore_errors=True)\n            os.makedirs(PASTA_ENTRADA)\n            excel = modelo = None\n            for nome, conteudo in _arquivos_enviados(upload.value):\n                destino = os.path.join(PASTA_ENTRADA, os.path.basename(nome))\n                with open(destino, "wb") as f:\n                    f.write(conteudo)\n                if nome.lower().endswith((".xlsx", ".csv")):\n                    excel = destino\n                elif nome.lower().endswith(".docx"):\n                    modelo = destino\n            if modelo is None:\n                if not modelo_padrao_b64:\n                    print("Envie também o modelo .docx.")\n                    return\n                modelo = os.path.join(PASTA_ENTRADA, "modelo_padrao.docx")\n                with open(modelo, "wb") as f:\n                    f.write(base64.b64decode(modelo_padrao_b64))\n            gerar_btn.disabled = True\n            gerar_btn.description = "Gerando..."\n            try:\n                estado["zip"] = g.gerar(excel, modelo, periodo.value.strip(), avaliadores.value.strip())\n            except Exception as e:  # mostra o erro de forma legível\n                print("\\nERRO:", e)\n                return\n            finally:\n                gerar_btn.disabled = False\n                gerar_btn.description = "2. Gerar documentos"\n            baixar_btn.disabled = False\n            # link de reserva, caso o download automático seja bloqueado pelo navegador\n            dados = base64.b64encode(open(estado["zip"], "rb").read()).decode()\n            nome_zip = "documentos_" + re.sub(r"[^\\w]+", "_", periodo.value.strip()).strip("_") + ".zip"\n            link.value = (f"<a download=\'{nome_zip}\' href=\'data:application/zip;base64,{dados}\' "\n                          f"style=\'font-size:13px\'>Se o download não começar, clique aqui.</a>")\n\n    def ao_baixar(_=None):\n        if estado["zip"] and baixar:\n            with log:\n                baixar(estado["zip"])\n\n    upload.observe(ao_escolher, names="value")\n    gerar_btn.on_click(ao_gerar)\n    baixar_btn.on_click(ao_baixar)\n\n    titulo = w.HTML("<h3 style=\'margin:4px 0;color:#0B2E59\'>Documentos de Acompanhamento Periódico</h3>"\n                    "<div style=\'color:#5B6B7F\'>Preencha, escolha o Excel exportado do BI e clique em gerar.</div>")\n    painel = w.VBox([titulo, periodo, avaliadores, w.HBox([upload, recebidos]), gerar_btn,\n                     w.HBox([baixar_btn, link]), log])\n    display(painel)\n    # devolvido para testes\n    return {"periodo": periodo, "avaliadores": avaliadores, "upload": upload, "gerar": ao_gerar,\n            "escolher": ao_escolher, "baixar": ao_baixar, "estado": estado, "log": log, "recebidos": recebidos}\n')
MODELO_PADRAO_B64 = '...'
try:
    from google.colab import files, output
    output.enable_custom_widget_manager()
    baixar = files.download
except ImportError:  # fora do Colab (Jupyter): só o link de download
    baixar = None
import importlib, gerar_documentos, painel
importlib.reload(gerar_documentos); importlib.reload(painel)
_painel = painel.criar_painel(MODELO_PADRAO_B64, baixar)


---
**Plano B** — só se o botão *Escolher Excel* do painel não funcionar: preencha os campos abaixo, clique no ▶ desta célula e escolha o Excel no botão que aparecer.

In [ ]:
#@title Plano B: enviar o Excel e gerar { display-mode: "form" }
PERIODO = ""  #@param {type:"string"}
AVALIADORES = ""  #@param {type:"string"}
import os, base64, shutil
from google.colab import files
import gerar_documentos
shutil.rmtree("entrada", ignore_errors=True); os.makedirs("entrada")
excel = modelo = None
for nome, dados in files.upload().items():
    destino = os.path.join("entrada", nome)
    open(destino, "wb").write(dados)
    if nome.lower().endswith((".xlsx", ".csv")): excel = destino
    elif nome.lower().endswith(".docx"): modelo = destino
if not modelo:
    modelo = "entrada/modelo_padrao.docx"
    open(modelo, "wb").write(base64.b64decode(MODELO_PADRAO_B64))
if not excel or not PERIODO.strip():
    raise SystemExit("Preencha o PERIODO e escolha o Excel exportado do BI.")
files.download(gerar_documentos.gerar(excel, modelo, PERIODO.strip(), AVALIADORES.strip()))
